In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By 
from selenium.webdriver.chrome.options import Options
import random # Rastgele gecikme için

# ----------------------------
# 1. PARAMETRELER VE AYARLAR
# ----------------------------
ANA_URL = "https://www.emlakjet.com/satilik-konut/izmir-bornova"
MAX_CEKIM_SAYISI = 5 # 🚨 Kaç sayfa çekileceği. Tekrar 1-5 aralığı için 5 yapıldı.
CIKTI_DOSYA_ADI = f'emlakjet_bornova_verisi_TIKLAMALI_FINAL.csv'

# İlan listeleme sayfasında tüm ilanları kapsayan ana div'in sınıfı
KAPSAYICI_SINIF = 'styles_wrapper__K8w6q' 
# 🚨 SONRAKİ SAYFA BUTONUNUN SINIFI (Sizin tespitiniz)
SONRAKİ_BUTON_SINIFI = 'ssssbtqkd_styles' 

# Detay Sayfası Sınıfları
FIYAT_DIV_SINIFI = 'styles_topWrap___T6DR'
FIYAT_SPAN_SINIFI = 'styles_price__6wmxk' 
ADRES_SPAN_SINIFI = 'styles_location__HguIg'
OZELIK_LISTESI_DIV_SINIFI = 'styles_inner__qKPCB' # UL'yi kapsayan div
ACIKLAMA_DIV_SINIFI = 'styles_wrapper__hycT_'

# ----------------------------
# 2. FONKSİYONLAR
# ----------------------------

def cerez_kabul_et(driver):
    """Emlakjet'teki çerez bildirimini (Kabul Et butonu) kapatmayı dener."""
    try:
        print("   -> Çerez pop-up'ı kontrol ediliyor...")
        kabul_butonu = driver.find_element(By.XPATH, "//button[text()='Kabul Et']")
        kabul_butonu.click()
        time.sleep(1)
        print("   -> Çerezler kabul edildi.")
        return True
    except NoSuchElementException:
        print("   -> Çerez pop-up'ı bulunamadı/zaten kapalı.")
        return False
    except Exception as e:
        print(f"   !!! Çerez kabul edilirken hata oluştu: {e}")
        return False

def ilan_linklerini_topla_TIKLAMALI(driver, url, max_sayfa):
    """URL değiştirmek yerine butona tıklayarak ilanları toplar."""
    print(f"📢 Tıklama Yöntemiyle Linkler Toplanıyor (Max {max_sayfa} sayfa)...")
    linkler = set()
    
    # Başlangıç Sayfasına Gitme
    driver.get(url) 
    time.sleep(random.uniform(5, 8))
    cerez_kabul_et(driver)
    time.sleep(2) 
    
    for sayac in range(1, max_sayfa + 1):
        rastgele_gecikme = random.uniform(8, 12) # Sayfalar arası uzun ve rastgele bekleme
        print(f"   -> Sayfa {sayac}: {driver.current_url} ({rastgele_gecikme:.2f} sn bekleniyor)")
        
        try:
            # Sayfayı yenilemeye zorla (önbellek sorununu aşmak için)
            driver.refresh()
            time.sleep(random.uniform(2, 4)) # Kısa yenileme sonrası bekleme
            
            # 1. HTML'i al ve linkleri topla
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            buyuk_kapsayici = soup.find('div', class_=KAPSAYICI_SINIF)
            
            tum_alt_linkler = []
            if buyuk_kapsayici:
                tum_alt_linkler = buyuk_kapsayici.find_all('a', href=True)
            
            link_sayisi = 0
            for link_etiketi in tum_alt_linkler:
                if link_etiketi['href'].startswith('/ilan/'):
                    tam_link = "https://www.emlakjet.com" + link_etiketi['href']
                    linkler.add(tam_link)
                    link_sayisi += 1
            
            print(f"   -> Sayfa {sayac}'tan {link_sayisi} yeni link bulundu. Toplam: {len(linkler)}")
            
            if sayac == max_sayfa:
                break 
            
            # 2. SONRAKİ SAYFA BUTONUNA TIKLAMA
            time.sleep(rastgele_gecikme) # Link toplama ve tıklama arasında bekle
            
            sonraki_buton = driver.find_element(By.CLASS_NAME, SONRAKİ_BUTON_SINIFI)
            
            # Pasif veya bulunamazsa döngüden çık
            if 'disabled' in sonraki_buton.get_attribute('class'):
                print("   -> Sonraki sayfa butonu pasif. Liste sonuna ulaşıldı.")
                break
                
            sonraki_buton.click() # Butona tıklama
            print("   -> Sonraki sayfaya tıklandı.")
            
        except NoSuchElementException:
            print("   -> Sonraki Sayfa butonu bulunamadı (NoSuchElementException). Liste sonu.")
            break
        except Exception as e:
            print(f"   -> Sayfa {sayac} işlenirken beklenmedik bir hata oluştu: {e}")
            break
            
    print(f"✅ Toplam {len(linkler)} adet benzersiz ilan linki bulundu.")
    return list(linkler)

def ilan_detaylarini_cek(soup, url):
    """İlanın detay sayfasından tüm özellikleri çeker."""
    detaylar = {'URL': url}
    
    # 1. Fiyat
    try:
        price_wrap = soup.find('div', class_=FIYAT_DIV_SINIFI)
        fiyat_etiketi = price_wrap.find('span', class_=FIYAT_SPAN_SINIFI)
        
        if not fiyat_etiketi:
             fiyat_etiketi = price_wrap.find('span') 
        
        if fiyat_etiketi:
            detaylar['Fiyat'] = fiyat_etiketi.text.strip().replace('\n', '').replace(' ', '')
        else:
            detaylar['Fiyat'] = 'N/A'
            
    except:
        detaylar['Fiyat'] = 'N/A'
        
    # Mahalle ve Adres Ayrıştırma
    try:
        mahalle_etiketi = soup.find('span', class_=ADRES_SPAN_SINIFI) 
        
        if mahalle_etiketi:
            tam_adres = mahalle_etiketi.text.strip()
            adres_parcalari = [p.strip() for p in tam_adres.split('-')]
            
            if len(adres_parcalari) >= 3:
                detaylar['il'] = adres_parcalari[0]    
                detaylar['Ilce'] = adres_parcalari[1]  
                detaylar['Mahalle'] = adres_parcalari[2]
            else:
                detaylar['Mahalle_Tam_Metin'] = tam_adres
                detaylar['il'] = 'N/A'
                detaylar['Ilce'] = 'N/A'
        else:
            detaylar['Mahalle'] = 'N/A'
            detaylar['il'] = 'N/A'
            detaylar['Ilce'] = 'N/A'
            
    except Exception as e:
        detaylar['il'] = 'HATA'
    
    # Başlık
    try:
        baslik_etiketi = soup.find('h1', class_='ej-title') 
        detaylar['Baslik'] = baslik_etiketi.text.strip() if baslik_etiketi else 'N/A'
    except:
        detaylar['Baslik'] = 'N/A'

    # 2. Key-Value Özellikleri (styles_inner__qKPCB ul li içinde)
    try:
        inner_div = soup.find('div', class_=OZELIK_LISTESI_DIV_SINIFI)
        ul_listesi = inner_div.find('ul')
        
        # Genel LI aramasına geri dönüldü
        ozellik_listesi = ul_listesi.find_all('li') if ul_listesi else []
            
        for item in ozellik_listesi:
            etiket = item.find('span') 
            deger = item.find_all('span')[-1]
            
            if etiket and deger:
                key = etiket.text.strip()
                value = deger.text.strip()
                detaylar[key] = value
                
    except Exception as e:
        pass
            
    # 3. İlan Açıklaması (styles_wrapper__hycT_ içinde)
    try:
        aciklama_divi = soup.find('div', class_=ACIKLAMA_DIV_SINIFI)
        if aciklama_divi:
            detaylar['Aciklama'] = aciklama_divi.text.strip().replace('\n', ' ')
        else:
            detaylar['Aciklama'] = 'N/A'
    except:
        detaylar['Aciklama'] = 'N/A'
        
    return detaylar

# ----------------------------
# 3. ANA İŞLEM AKIŞI
# ----------------------------

if __name__ == "__main__":
    
    driver = None
    try:
        print("🚀 Chrome WebDriver başlatılıyor (Gizli Mod)...")
        
        # 🚨 GİZLENME VE BAŞSIZ MOD AYARLARI 🚨
        options = Options()
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        options.add_argument("--headless") 
        options.add_argument("window-size=1920,1080")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
        
        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        # 1. Belirtilen sayfa sayısınca butona tıklayarak linkleri toplama
        ilan_linkleri = ilan_linklerini_topla_TIKLAMALI(driver, ANA_URL, MAX_CEKIM_SAYISI)
        
        tum_ilan_verileri = []
        
        # 2. Her bir ilan linkini ziyaret etme ve detayları çekme
        print("\n🔍 Detay sayfaları ziyaret edilerek veri çekiliyor...")
        
        for i, link in enumerate(ilan_linkleri):
            print(f"   -> İlan {i+1}/{len(ilan_linkleri)}")
            
            # Rastgele bekleme (İlanlar arası gizlenme)
            time.sleep(random.uniform(5, 10)) 
            
            try:
                driver.get(link)
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                ilan_detaylari = ilan_detaylarini_cek(soup, link)
                tum_ilan_verileri.append(ilan_detaylari)
                
            except TimeoutException:
                print(f"   !!! ZAMAN AŞIMI HATASI: {link} - Sayfa yüklenemedi.")
                continue
            except Exception as e:
                print(f"   -> HATA: {link} adresinde beklenmedik bir hata oluştu: {e}")
                continue 
                
        # 3. Verileri Kaydetme
        if tum_ilan_verileri:
            print("\n💾 Veriler DataFrame'e dönüştürülüp CSV olarak kaydediliyor...")
            df = pd.DataFrame(tum_ilan_verileri)
            
            df.to_csv(CIKTI_DOSYA_ADI, index=False, encoding='utf-8-sig')
            print(f"✅ İŞLEM TAMAMLANDI! Veriler '{CIKTI_DOSYA_ADI}' dosyasına kaydedildi.")
        else:
            print("❌ Veri çekme başarısız oldu. Kaydedilecek veri yok.")
            
    except Exception as e:
        print(f"\nFATAL HATA: WebDriver veya ana akış başlatılamadı: {e}")
        
    finally:
        if driver:
            driver.quit()

🚀 Chrome WebDriver başlatılıyor (Gizli Mod)...
📢 Tıklama Yöntemiyle Linkler Toplanıyor (Max 5 sayfa)...
   -> Çerez pop-up'ı kontrol ediliyor...
   -> Çerez pop-up'ı bulunamadı/zaten kapalı.
   -> Sayfa 1: https://www.emlakjet.com/satilik-konut/izmir-bornova (8.38 sn bekleniyor)
   -> Sayfa 1'tan 30 yeni link bulundu. Toplam: 30
   -> Sonraki sayfaya tıklandı.
   -> Sayfa 2: https://www.emlakjet.com/satilik-konut/izmir-bornova?sayfa=2 (10.84 sn bekleniyor)
   -> Sayfa 2'tan 30 yeni link bulundu. Toplam: 60
   -> Sonraki sayfaya tıklandı.
   -> Sayfa 3: https://www.emlakjet.com/satilik-konut/izmir-bornova?sayfa=3 (9.11 sn bekleniyor)
   -> Sayfa 3'tan 30 yeni link bulundu. Toplam: 90
   -> Sonraki sayfaya tıklandı.
   -> Sayfa 4: https://www.emlakjet.com/satilik-konut/izmir-bornova?sayfa=4 (9.44 sn bekleniyor)
   -> Sayfa 4'tan 30 yeni link bulundu. Toplam: 120
   -> Sonraki sayfaya tıklandı.
   -> Sayfa 5: https://www.emlakjet.com/satilik-konut/izmir-bornova?sayfa=5 (11.80 sn bekleniyo

In [ ]:
import pandas as pd 
dataFrame6=pd.read_csv("emlakjet_bornova_verisi_TIKLAMALI_FINAL.csv")
dataFrame6.head(50)


,URL,Fiyat,il,Ilce,Mahalle,Baslik,İlan Numarası,İlan Güncelleme Tarihi,Türü,Kategorisi,...,WC Metrekare,Parsel,Görüntülü Gezilebilir mi?,Zemin Etüdü,Balkon Sayısı,Ada,Balkon Tipi,Balkon Metrekare,Pafta,Kattaki Daire Sayısı
0,https://www.emlakjet.com/ilan/onu-cok-acik-fer...,3.695.000TL,İzmir,Bornova,Kazımdirik Mahallesi,NaN,18461450.0,21 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.emlakjet.com/ilan/luks-yapili-manz...,7.500.000TL,İzmir,Bornova,Atatürk Mahallesi,NaN,18322222.0,18 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.emlakjet.com/ilan/tasinmaya-hazir-...,4.650.000TL,İzmir,Bornova,Kazımdirik Mahallesi,NaN,18437428.0,17 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.emlakjet.com/ilan/bornova-goldiva-...,10.900.000TL,İzmir,Bornova,Rafet Paşa Mahallesi,NaN,18411947.0,27 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.emlakjet.com/ilan/bornova-pinarbas...,8.149.000TL,İzmir,Bornova,Kemalpaşa Mahallesi,NaN,18424391.0,15 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,https://www.emlakjet.com/ilan/folkart-incity-h...,8.250.000TL,İzmir,Bornova,Kazımdirik Mahallesi,NaN,18393535.0,10 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,https://www.emlakjet.com/ilan/bornova-evka-3-t...,3.975.000TL,İzmir,Bornova,Evka 3 Mahallesi,NaN,18492989.0,26 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,https://www.emlakjet.com/ilan/deniz-ve-sehir-m...,10.800.000TL,İzmir,Bornova,Atatürk Mahallesi,NaN,18360473.0,06 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,https://www.emlakjet.com/ilan/bornova-goldiva-...,10.750.000TL,İzmir,Bornova,Rafet Paşa Mahallesi,NaN,18466540.0,27 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,https://www.emlakjet.com/ilan/altindagda-ana-c...,2.850.000TL,İzmir,Bornova,Serintepe Mahallesi,NaN,18479730.0,24 Kasım 2025,Konut,Satılık,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
dataFrameSon2=pd.read_csv('emlakjet_bornova_verisi_S06.csv')

In [13]:
dataFrameSon=pd.read_csv( "emlakjet_bornova_verisi_TIKLAMALI_FINAL.csv")

In [30]:
arr = dataFrameSon["İlan Numarası"].unique()
arr.sort()
print(arr)

[18322222. 18326254. 18326258. 18327142. 18327367. 18327369. 18327373.
 18327392. 18327400. 18328635. 18328899. 18329259. 18330464. 18330482.
 18334175. 18335235. 18335585. 18335815. 18336228. 18337379. 18343196.
 18344053. 18345471. 18345678. 18346304. 18348049. 18349459. 18350160.
 18350338. 18350339. 18354131. 18354669. 18355391. 18356018. 18360473.
 18361744. 18362868. 18362869. 18363102. 18369316. 18372165. 18372791.
 18377177. 18379581. 18379703. 18382292. 18383877. 18384284. 18385401.
 18389321. 18389324. 18393535. 18394527. 18394890. 18394974. 18395920.
 18398032. 18400657. 18403037. 18403047. 18403049. 18404922. 18404956.
 18404963. 18411934. 18411947. 18413258. 18413480. 18413630. 18415790.
 18416215. 18417743. 18418866. 18419022. 18419522. 18421152. 18421508.
 18423343. 18424391. 18428463. 18430224. 18430383. 18433813. 18437428.
 18438898. 18439218. 18439559. 18440228. 18441103. 18441481. 18441503.
 18442128. 18443930. 18444227. 18444606. 18446715. 18449725. 18450699.
 18451